In [ ]:
%pip install 'geobrix[vizx,light_env6] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl'

In [ ]:
%restart_python

In [ ]:
import pathlib
import matplotlib
matplotlib.use("Agg")

from databricks.labs.gbx import vizx as vz

LAZ = "/Volumes/geospatial_docs/wireless_coverage/data/lidar/sf/laz/ARRA_CA_GOLDENGATE_2010/0-0-0-0.laz"
WFS_OUT_DIR = "/Workspace/Users/mjohns@databricks.com/geobrix_spikes/show_flythrough_smoke/resources/geobrix"
MP4_VOL_DIR = "/Volumes/geospatial_docs/geobrix/sample-data/show_flythrough_smoke"

print(f"[smoke] source LAZ: {LAZ}")
assert pathlib.Path(LAZ).exists(), f"LAZ not found: {LAZ}"

# --- Smoke: persist=True with GIF + MP4 + volume copy ---
result = vz.show_flythrough(
    LAZ,
    out_dir=WFS_OUT_DIR,
    mp4=True,
    mp4_volume_dir=MP4_VOL_DIR,
    preview_mp4=True,
    overwrite=True,  # force fresh render for the smoke
    fps=5,
    seconds=6,
    max_points=80_000,
)

gif_path = result["gif"]
mp4_path = result["mp4"]
snippet = result["md_snippet"]
preview_html = result["_preview_html"]

print(f"[smoke] GIF  path : {gif_path}")
print(f"[smoke] MP4  path : {mp4_path}")
print(f"[smoke] md_snippet: {snippet}")

# --- GIF assertions ---
assert gif_path is not None and gif_path.exists(), "GIF was not written"
gif_size = gif_path.stat().st_size
print(f"[smoke] GIF  size : {gif_size:,} bytes ({gif_size/1024:.1f} KB)")
assert gif_size > 100_000, f"GIF looks like a stub: {gif_size} bytes (expected >100 KB)"

# Verify GIF is decodable
from PIL import Image
with Image.open(str(gif_path)) as img:
    n_frames = img.n_frames
print(f"[smoke] GIF  frames: {n_frames}")
assert n_frames >= 5, f"Expected >=5 frames, got {n_frames}"

# --- MP4 assertions ---
assert mp4_path is not None and mp4_path.exists(), "MP4 was not written"
mp4_size = mp4_path.stat().st_size
print(f"[smoke] MP4  size : {mp4_size:,} bytes ({mp4_size/1024:.1f} KB)")
assert mp4_size > 1_000, f"MP4 looks like a stub: {mp4_size} bytes"

# --- Volume copy ---
vol_mp4 = pathlib.Path(MP4_VOL_DIR) / "0-0-0-0.mp4"
assert vol_mp4.exists(), f"MP4 was not copied to volume: {vol_mp4}"
print(f"[smoke] Vol MP4 : {vol_mp4} ({vol_mp4.stat().st_size:,} bytes)")

# --- preview_html ---
assert preview_html is not None, "_preview_html was not set (expected embed)"
assert "data:video/mp4;base64," in preview_html, "preview_html does not contain base64 video"
print(f"[smoke] preview_html length: {len(preview_html)} chars")

# --- md_snippet ---
assert snippet is not None, "md_snippet should be non-None with persist=True"
print(f"[smoke] PASSED — GIF {gif_size:,}B  MP4 {mp4_size:,}B  WFS-write clean")
